# Garbage Classification using Keras and Transfer Learning (v2)
Based on the Kaggle notebook by Mostafa Mohamed (Apache 2.0):
https://www.kaggle.com/code/mostafaabla/garbage-classification-keras-transfer-learning

Updated version. **Not yet run end to end.** See README for the list of changes.

# Setup

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import xception
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Configuration

In [ ]:
IMAGE_SIZE = (320, 320)
BATCH_SIZE = 64
EPOCHS = 20
FINE_TUNE_EPOCHS = 5
BASE_PATH = "garbage_classification/"
CLASSES = ['paper','cardboard','plastic','metal','trash','battery',
           'shoes','clothes','green-glass','brown-glass','white-glass','biological']

# Build the DataFrame

In [ ]:
rows = []
for c in CLASSES:
    for f in os.listdir(os.path.join(BASE_PATH, c)):
        rows.append((f"{c}/{f}", c))
df = pd.DataFrame(rows, columns=['filename','category']).sample(frac=1, random_state=SEED).reset_index(drop=True)
print('number of images =', len(df))
df['category'].value_counts().plot.bar()
plt.title('Images per class'); plt.show()

# Split: 80% train, 10% validation, 10% test

In [ ]:
train_df, rest_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['category'])
val_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=SEED, stratify=rest_df['category'])
print('train', len(train_df), 'validation', len(val_df), 'test', len(test_df))

# Data generators (augmentation on for training only)

In [ ]:
def flow(gen, d, shuffle=True, bs=BATCH_SIZE):
    return gen.flow_from_dataframe(d, BASE_PATH, x_col='filename', y_col='category',
        target_size=IMAGE_SIZE, class_mode='categorical', batch_size=bs, shuffle=shuffle, classes=sorted(CLASSES))

train_gen = flow(ImageDataGenerator(rotation_range=20, zoom_range=0.2, width_shift_range=0.1,
                 height_shift_range=0.1, horizontal_flip=True), train_df)
val_gen = flow(ImageDataGenerator(), val_df, shuffle=False)
test_gen = flow(ImageDataGenerator(), test_df, shuffle=False)

# Model: frozen Xception + classification head

In [ ]:
base = xception.Xception(include_top=False, weights='imagenet', input_shape=IMAGE_SIZE + (3,))
base.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,))
x = layers.Lambda(xception.preprocess_input)(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(CLASSES), activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.compile(loss='categorical_crossentropy', optimizer=keras.optimizers.Adam(1e-3),
              metrics=['categorical_accuracy'])
model.summary()

# Train (stage 1: head only)

In [ ]:
early_stop = EarlyStopping(patience=2, verbose=1, monitor='val_categorical_accuracy',
                           mode='max', min_delta=0.001, restore_best_weights=True)
history = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=[early_stop])

# Optional stage 2: fine-tune the top of Xception with a low learning rate

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(loss='categorical_crossentropy', optimizer=keras.optimizers.Adam(1e-5),
              metrics=['categorical_accuracy'])
history_ft = model.fit(train_gen, epochs=FINE_TUNE_EPOCHS, validation_data=val_gen, callbacks=[early_stop])
model.save('garbage_classifier.keras')

# Training curves

In [ ]:
acc = history.history['categorical_accuracy'] + history_ft.history['categorical_accuracy']
val = history.history['val_categorical_accuracy'] + history_ft.history['val_categorical_accuracy']
plt.plot(acc, label='train'); plt.plot(val, label='validation')
plt.axvline(len(history.history['loss']) - 1, ls='--', c='gray', label='fine-tuning starts')
plt.legend(); plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.show()

# Evaluate on the test set

In [ ]:
_, acc = model.evaluate(test_gen)
print('test accuracy =', round(acc * 100, 2), '%')

probs = model.predict(test_gen)
y_pred = probs.argmax(1)
y_true = test_gen.classes
names = list(test_gen.class_indices.keys())
print(classification_report(y_true, y_pred, target_names=names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=names, yticklabels=names, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()